# Name-Me Data Explorer

This notebook explores:
1. **Current DB** - Names already in the database
2. **genderNamesITA** - 32K+ Italian names with gender frequency
3. **names-dataset** - 730K names from Facebook data (106 countries)
4. **popular-names-by-country** - Popular names by country (CC0 license)

In [1]:
import sys
import os

# Add backend to path
backend_path = os.path.join(os.getcwd(), 'backend')
sys.path.insert(0, backend_path)

# Change to backend directory for relative imports and .env loading
os.chdir(backend_path)

print(f"Python: {sys.executable}")
print(f"Working dir: {os.getcwd()}")

Python: /Users/nicolamacchitella/Documents/name-me/backend/venv/bin/python
Working dir: /Users/nicolamacchitella/Documents/name-me/backend


In [2]:
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker

# Use synchronous SQLite for easier notebook exploration
DATABASE_PATH = "data/baby_names.db"
engine = create_engine(f"sqlite:///{DATABASE_PATH}", echo=False)
Session = sessionmaker(bind=engine)

print(f"Connected to: {DATABASE_PATH}")

Connected to: data/baby_names.db


## Explore Tables

In [3]:
# List all tables
with engine.connect() as conn:
    result = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table'"))
    tables = [row[0] for row in result]
    print("Tables:", tables)

Tables: ['alembic_version', 'couples', 'users', 'user_preferences', 'invites', 'names', 'swipes']


## Query Names

In [4]:
from app.models.name import Name

with Session() as session:
    # Get total count
    total = session.query(Name).count()
    print(f"Total names: {total}")
    
    # Sample names by gender
    for gender in ['M', 'F', 'U']:
        count = session.query(Name).filter(Name.gender == gender).count()
        print(f"  {gender}: {count}")

Total names: 60
  M: 29
  F: 30
  U: 1


In [5]:
# Show sample names
with Session() as session:
    names = session.query(Name).limit(10).all()
    for n in names:
        print(f"{n.name} ({n.gender}) - {n.origin}: {n.meaning}")

Marco (M) - italian: Warlike, dedicated to Mars
Luca (M) - italian: Bringer of light
Alessandro (M) - italian: Defender of the people
Francesco (M) - italian: Free man
Lorenzo (M) - italian: From Laurentum
Matteo (M) - italian: Gift of God
Leonardo (M) - italian: Brave lion
Andrea (U) - italian: Brave, manly
Gabriele (M) - italian: God is my strength
Riccardo (M) - italian: Powerful ruler


## Names by Origin

In [6]:
from sqlalchemy import func

with Session() as session:
    origins = session.query(
        Name.origin,
        func.count(Name.id).label('count')
    ).group_by(Name.origin).order_by(func.count(Name.id).desc()).all()
    
    print("Names by origin:")
    for origin, count in origins:
        print(f"  {origin}: {count}")

Names by origin:
  english: 20
  irish: 20
  italian: 20


## Query Users & Couples

In [7]:
from app.models.user import User
from app.models.couple import Couple

with Session() as session:
    users = session.query(User).all()
    couples = session.query(Couple).all()
    
    print(f"Users: {len(users)}")
    print(f"Couples: {len(couples)}")
    
    for user in users[:5]:
        print(f"  - {user.display_name} ({user.email})")

Users: 1
Couples: 1
  - Nicola Macchitella (nicola.macchitella@gmail.com)


## Query Swipes

In [8]:
from app.models.swipe import Swipe

with Session() as session:
    swipes = session.query(Swipe).all()
    likes = [s for s in swipes if s.liked]
    
    print(f"Total swipes: {len(swipes)}")
    print(f"Likes: {len(likes)}")
    print(f"Passes: {len(swipes) - len(likes)}")

Total swipes: 0
Likes: 0
Passes: 0


---
# External Datasets Exploration

In [9]:
import pandas as pd
import os

# Set paths relative to project root
PROJECT_ROOT = os.path.dirname(os.getcwd())  # Go up from backend/
RAW_DATA = os.path.join(PROJECT_ROOT, "data", "raw")
print(f"Raw data dir: {RAW_DATA}")

Raw data dir: /Users/nicolamacchitella/Documents/name-me/data/raw


## 1. genderNamesITA (Italian Names with Gender)
Source: Italian registry data 1985-2014, 32K+ names

In [18]:
# Load Italian names dataset
ita_names = pd.read_csv(os.path.join(RAW_DATA, "gender_firstnames_ITA.csv"))
print(f"Shape: {ita_names.shape}")
print(f"Columns: {list(ita_names.columns)}")
ita_names.head(10)

Shape: (32844, 4)
Columns: ['nome', 'tot', 'male', 'female']


,nome,tot,male,female
0,A IPPOLITO,5,5,0
1,A MARIA,5,0,5
2,A. ANGELO,3,3,0
3,A. GIUSEPPE,1,1,0
4,A. MARIA,5,0,5
5,A. ORESTE,3,3,0
6,A. VITTORIO,5,5,0
7,A.M.LODOVICA,5,0,5
8,A.R. MARIA,5,0,5
9,A.RAFFAELE,10,10,0


In [19]:
# Analyze gender distribution
# male > female = likely male, female > male = likely female
ita_names['gender'] = ita_names.apply(
    lambda r: 'M' if r['male'] > r['female'] else ('F' if r['female'] > r['male'] else 'U'), 
    axis=1
)
print("Gender distribution:")
print(ita_names['gender'].value_counts())
print(f"\nTotal unique names: {len(ita_names)}")

# Show top names by frequency
print("\nTop 20 names by total occurrences:")
ita_names.nlargest(20, 'tot')[['nome', 'tot', 'male', 'female', 'gender']]

Gender distribution:
gender
M    23912
F     8917
U       15
Name: count, dtype: int64

Total unique names: 32844

Top 20 names by total occurrences:


,nome,tot,male,female,gender
15446,GIUSEPPE,191186,191140,46,M
3090,ANTONIO,136765,136765,0,M
14781,GIOVANNI,128981,128950,31,M
11960,FRANCESCO,108184,108184,0,M
19376,LUIGI,85747,85742,5,M
21459,MARIO,84733,84733,0,M
27638,ROBERTO,70832,70832,0,M
25057,PAOLO,66776,66776,0,M
2092,ANGELO,61193,61188,5,M
12239,FRANCO,56896,56896,0,M


## 2. Philippe Remy's names-dataset
730K first names from Facebook data across 106 countries

In [20]:
from names_dataset import NameDataset

# Load the names dataset (this may take a moment)
nd = NameDataset()

# Get Italian names
italy_first_names = nd.get_top_names(n=100, country_alpha2='IT')
print("Top Italian names from names-dataset:")
print(f"Keys returned: {italy_first_names.keys()}")

# Access based on actual keys
if 'Male' in italy_first_names:
    print(f"  Male: {italy_first_names['Male'][:10]}")
    print(f"  Female: {italy_first_names['Female'][:10]}")
elif 'M' in italy_first_names:
    print(f"  Male: {italy_first_names['M'][:10]}")
    print(f"  Female: {italy_first_names['F'][:10]}")
else:
    print(italy_first_names)

Top Italian names from names-dataset:
Keys returned: dict_keys(['IT'])
{'IT': {'M': ['Giuseppe', 'Francesco', 'Marco', 'Andrea', 'Antonio', 'Alessandro', 'Luca', 'Giovanni', 'Roberto', 'Stefano', 'Paolo', 'Michele', 'Salvatore', 'Davide', 'Matteo', 'Fabio', 'Vincenzo', 'Luigi', 'Mario', 'Massimo', 'Simone', 'Daniele', 'Angelo', 'Nicola', 'Lorenzo', 'Claudio', 'Maurizio', 'Domenico', 'Franco', 'Alberto', 'Riccardo', 'Mauro', 'Carlo', 'Alessio', 'Pietro', 'Federico', 'Gabriele', 'Gianluca', 'Giorgio', 'Gianni', 'Pasquale', 'Fabrizio', 'Enrico', 'Mattia', 'Emanuele', 'Raffaele', 'Sergio', 'Filippo', 'Giacomo', 'Enzo', 'Cristian', 'Luciano', 'Leonardo', 'Massimiliano', 'Gaetano', 'Bruno', 'Dario', 'Diego', 'Alex', 'Vito', 'Christian', 'Mirko', 'Ivan', 'Carmine', 'Manuel', 'Tommaso', 'Piero', 'Ciro', 'Giancarlo', 'Marcello', 'Vittorio', 'Danilo', 'Valerio', 'Giulio', 'Gennaro', 'Carmelo', 'Rocco', 'Aldo', 'Antonino', 'Gianfranco', 'Sandro', 'Renato', 'Salvo', 'Samuele', 'Edoardo', 'Daniel',

In [21]:
# Search for a specific name and get gender probability
test_names = ['Marco', 'Giulia', 'Andrea', 'Luca', 'Sofia']
print("Name search results:")
for name in test_names:
    result = nd.search(name)
    if result:
        gender = result.get('gender', {})
        country = result.get('country', {})
        print(f"  {name}: gender={gender}, top_country={list(country.items())[:3] if country else 'N/A'}")

Name search results:
  Marco: gender={}, top_country=N/A
  Giulia: gender={}, top_country=N/A
  Andrea: gender={}, top_country=N/A
  Luca: gender={}, top_country=N/A
  Sofia: gender={}, top_country=N/A


## 3. Popular Names by Country (CC0 License)
Curated list of popular forenames and surnames by country

In [22]:
# Load popular names by country
forenames_path = os.path.join(RAW_DATA, "popular-names-by-country-dataset-main", "common-forenames-by-country.csv")
pop_names = pd.read_csv(forenames_path)
print(f"Shape: {pop_names.shape}")
print(f"Columns: {list(pop_names.columns)}")
print(f"\nCountries: {pop_names['Country'].nunique()}")
pop_names.head()

Shape: (2480, 12)
Columns: ['Country', 'Country Group', 'Region', 'Population', 'Note', 'Year', 'Romanization', 'Index', 'Name Group', 'Gender', 'Localized Name', 'Romanized Name']

Countries: 106


,Country,Country Group,Region,Population,Note,Year,Romanization,Index,Name Group,Gender,Localized Name,Romanized Name
0,AD,1,NaN,NaN,NaN,2018.0,N,1,N-AD-1-F-1,F,Martina,Martina
1,AD,1,NaN,NaN,NaN,2018.0,N,2,N-AD-1-F-2,F,Emma,Emma
2,AD,1,NaN,NaN,NaN,2018.0,N,3,N-AD-1-F-3,F,Jana,Jana
3,AD,1,NaN,NaN,NaN,2018.0,N,4,N-AD-1-F-4,F,Lucia,Lucia
4,AD,1,NaN,NaN,NaN,2018.0,N,1,N-AD-1-M-1,M,Iker,Iker


In [23]:
# Filter for Italian names (IT)
italy_pop = pop_names[pop_names['Country'] == 'IT']
print(f"Italian names in dataset: {len(italy_pop)}")
print(f"\nMale names: {len(italy_pop[italy_pop['Gender'] == 'M'])}")
print(f"Female names: {len(italy_pop[italy_pop['Gender'] == 'F'])}")

print("\nTop Italian names:")
italy_pop[['Romanized Name', 'Gender', 'Index']].head(20)

Italian names in dataset: 20

Male names: 10
Female names: 10

Top Italian names:


,Romanized Name,Gender,Index
1156,Sofia,F,1
1157,Aurora,F,2
1158,Giulia,F,3
1159,Ginevra,F,4
1160,Beatrice,F,5
1161,Alice,F,6
1162,Vittoria,F,7
1163,Emma,F,8
1164,Ludovica,F,9
1165,Matilde,F,10


---
## Dataset Comparison Summary

In [24]:
# Summary comparison
print("=" * 60)
print("DATASET COMPARISON")
print("=" * 60)

print("\n1. genderNamesITA")
print(f"   - Total names: {len(ita_names):,}")
print(f"   - Has gender frequency data (male/female counts)")
print(f"   - Source: Italian registry 1985-2014")
print(f"   - Best for: Italian names with gender confidence scores")

print("\n2. names-dataset (Philippe Remy)")
print(f"   - Total names: 730K+ first names")
print(f"   - 106 countries with gender probabilities")
print(f"   - Source: Facebook data")
print(f"   - Best for: International names, gender inference")

print("\n3. popular-names-by-country")
print(f"   - Total entries: {len(pop_names):,}")
print(f"   - Countries: {pop_names['Country'].nunique()}")
print(f"   - Italian entries: {len(italy_pop)}")
print(f"   - License: CC0 (public domain)")
print(f"   - Best for: Popular/trending names by country")

print("\n" + "=" * 60)
print("RECOMMENDATION:")
print("=" * 60)
print("- Use genderNamesITA for comprehensive Italian names")
print("- Use names-dataset for gender inference API")
print("- Use popular-names-by-country for multi-country expansion")

DATASET COMPARISON

1. genderNamesITA
   - Total names: 32,844
   - Has gender frequency data (male/female counts)
   - Source: Italian registry 1985-2014
   - Best for: Italian names with gender confidence scores

2. names-dataset (Philippe Remy)
   - Total names: 730K+ first names
   - 106 countries with gender probabilities
   - Source: Facebook data
   - Best for: International names, gender inference

3. popular-names-by-country
   - Total entries: 2,480
   - Countries: 106
   - Italian entries: 20
   - License: CC0 (public domain)
   - Best for: Popular/trending names by country

RECOMMENDATION:
- Use genderNamesITA for comprehensive Italian names
- Use names-dataset for gender inference API
- Use popular-names-by-country for multi-country expansion
